In [0]:

catalog = dbutils.widgets.get("catalog")

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalog}.gold.dim_customers
USING DELTA
AS
SELECT DISTINCT
    customer_id,
    name,
    state,
    city,
    ingestion_timestamp
FROM ${catalog}.silver.customers;

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalog}.gold.dim_products
USING DELTA
AS
SELECT DISTINCT
    product_id,
    product_name,
    category,
    price,
    ingestion_timestamp
FROM ${catalog}.silver.products;

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalog}.gold.fact_sales
USING DELTA
AS
SELECT
    oc.customer_id,
    oc.name,
    oc.city,
    oc.state,
    oc.order_id,
    oc.order_date,
    oc.order_status,
    op.product_id,
    op.product_name,
    op.order_item_id,
    op.category,
    op.quantity,
    op.price,
    (cast(op.quantity as int) * cast(op.price as decimal(10,2))) AS revenue
FROM ${catalog}.silver.customer_orders oc
JOIN ${catalog}.silver.order_products op
    ON op.order_id = oc.order_id;